# Step 1: Data Preprocessing

In [27]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

Load Dataset

In [28]:
df1 = pd.read_csv("2.csv")

Drop irrelevant columns

In [29]:
df = df1.drop(["id", "first_name", "last_name", "email", "part_time_job", "absence_days", "weekly_self_study_hours"], axis=1)

Encode binary categorical features

In [30]:
df["gender"] = df["gender"].map({"male": 0, "female": 1})
df["extracurricular_activities"] = df["extracurricular_activities"].astype(int)

Encode target variable ("career_aspiration")

In [31]:
le_aspiration = LabelEncoder()
df["career_aspiration"] = le_aspiration.fit_transform(df["career_aspiration"])

Define features (X2) and target (y2)


In [32]:
x = df.drop("career_aspiration", axis=1)
y = df["career_aspiration"]

# Step 2: Split Dataset

In [33]:
from sklearn.model_selection import train_test_split

Split Dataset 2 into train/test sets

In [34]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Step 3: Train Model

## Decision Tree

In [35]:
from sklearn.tree import DecisionTreeClassifier

In [36]:
dt_academic = DecisionTreeClassifier(class_weight='balanced',max_depth=5, random_state=42)
dt_academic.fit(x_train, y_train)

DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=42)

Evaluate

In [37]:
print("Academic DT Accuracy:", dt_academic.score(x_test, y_test))

Academic DT Accuracy: 0.27


## Random Forest

In [38]:
from sklearn.ensemble import RandomForestClassifier

Train Model

In [ ]:
# Initialize and fit the label encoder
y_train_encoded = le_aspiration.fit_transform(y_train)  # y_train contains original career names

In [ ]:
from sklearn.model_selection import GridSearchCV

# GridSearch: It creates a "grid" of all possible parameter combinations we give it.

# CV: Stands for Cross-Validation. This is a technique to ensure the model's performance is tested reliably and doesn't just get lucky on one specific split of the data.

params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 6, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']  # Better feature sampling
}

grid = GridSearchCV(RandomForestClassifier(class_weight='balanced'),
                   params, cv=5, n_jobs=-1)
grid.fit(x_train, y_train_encoded)

best_model = grid.best_estimator_
print(f"Best Accuracy: {grid.best_score_:.3f}")

Best Accuracy: 0.917


# Step 4: Save Models & Label Encoders

In [ ]:
# import joblib

In [ ]:
# joblib.dump(model_academic, "model_academic.pkl")


In [ ]:
# joblib.dump(le_aspiration, "label_encoder_academic.pkl")  # For decoding career_aspiration

['label_encoder_academic.pkl']

In [ ]:
import joblib

# Save the best model
joblib.dump(best_model, 'best_rf_model.pkl')

# Save the label encoder (critical for decoding predictions)
joblib.dump(le_aspiration, 'label_encoder.pkl')

# Optional: Save the complete GridSearchCV object
joblib.dump(grid, 'grid_search_results.pkl')

['grid_search_results.pkl']

In [ ]:
def predict_career(user_input):
    # Load model and encoder
    model_academic = joblib.load("best_rf_model.pkl")
    le_academic = joblib.load("label_encoder.pkl")

    # Create DataFrame with correct feature names
    academic_features = pd.DataFrame([[
        user_input["gender"],
        user_input["extracurricular_activities"],
        user_input["math_score"],
        user_input["history_score"],
        user_input["physics_score"],
        user_input["chemistry_score"],
        user_input["biology_score"],
        user_input["english_score"],
        user_input["geography_score"]
    ]], columns=[
        'gender', 'extracurricular_activities', 'math_score',
        'history_score', 'physics_score', 'chemistry_score',
        'biology_score', 'english_score', 'geography_score'
    ])

    # Predict
    career_academic = le_academic.inverse_transform(
        model_academic.predict(academic_features)
    )[0]

    return {"academic_based_recommendation": career_academic}

In [ ]:
user_input = {
    "gender": 1,                        # 1=female
    "extracurricular_activities": 0,    # 1=True
    "math_score": 81,
    "history_score": 97,
    "physics_score": 95,
    "chemistry_score": 96,
    "biology_score": 65,
    "english_score": 77,
    "geography_score": 94
}

prediction = predict_career(user_input)
print(prediction)

{'academic_based_recommendation': 'Government Officer'}


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

# 1. Data Loading and Preparation
df = pd.read_csv("/2.csv")

# Drop irrelevant columns and encode
df = df.drop(["id", "first_name", "last_name", "email",
              "part_time_job", "absence_days", "weekly_self_study_hours"], axis=1)
df["gender"] = df["gender"].map({"male": 0, "female": 1})
df["extracurricular_activities"] = df["extracurricular_activities"].astype(int)

# 2. Feature Engineering
df['stem_score'] = (df['math_score'] + df['physics_score'] + df['chemistry_score']) / 3
df['humanities_score'] = (df['history_score'] + df['english_score']) / 2

# 3. Train-Test Split
x = df.drop("career_aspiration", axis=1)
y = df["career_aspiration"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 4. Label Encoding
le_aspiration = LabelEncoder()
y_train_encoded = le_aspiration.fit_transform(y_train)
y_test_encoded = le_aspiration.transform(y_test)  # Critical!

# 5. Model Training with Validation
# Create temporary holdout set from training data
x_temp, x_holdout, y_temp, y_holdout = train_test_split(
    x_train, y_train_encoded, test_size=0.2, random_state=42)

params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced'),
    params,
    cv=5,
    n_jobs=-1
)
grid.fit(x_temp, y_temp)

# 6. Evaluation
best_model = grid.best_estimator_
print("\nBest Parameters:", grid.best_params_)
print("Holdout Accuracy:", best_model.score(x_holdout, y_holdout))
print("Test Accuracy:", best_model.score(x_test, y_test_encoded))

# 7. Save Models
joblib.dump(best_model, 'best_rf_model.pkl')
joblib.dump(le_aspiration, 'label_encoder.pkl')

# 8. Detailed Analysis
print("\nFeature Importances:")
print(pd.DataFrame({
    'Feature': x_train.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False))

print("\nClassification Report:")
print(classification_report(
    y_test_encoded,
    best_model.predict(x_test),
    target_names=le_aspiration.classes_
))


Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 200}
Holdout Accuracy: 0.925
Test Accuracy: 0.9125

Feature Importances:
                       Feature  Importance
6                biology_score    0.124583
2                   math_score    0.118461
9                   stem_score    0.111759
7                english_score    0.102839
4                physics_score    0.102741
10            humanities_score    0.101253
3                history_score    0.101163
5              chemistry_score    0.097801
8              geography_score    0.097355
0                       gender    0.024828
1   extracurricular_activities    0.017217

Classification Report:
                       precision    recall  f1-score   support

           Accountant       0.96      0.93      0.94        81
               Artist       1.00      0.74      0.85        42
               Banker       0.97      0.91      0.94       101
       Business Owner       0.93

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier

# Load and prepare data
print("=== Data Loading ===")
df = pd.read_csv("/2.csv")
df = df.drop(["id", "first_name", "last_name", "email", "part_time_job", "absence_days", "weekly_self_study_hours"], axis=1)
df["gender"] = df["gender"].map({"male": 0, "female": 1})
df["extracurricular_activities"] = df["extracurricular_activities"].astype(int)

# Add composite features
print("\n=== Feature Engineering ===")
df['stem_score'] = (df['math_score'] + df['physics_score'] + df['chemistry_score']) / 3
df['humanities_score'] = (df['history_score'] + df['english_score']) / 2

# Split data
print("\n=== Data Splitting ===")
x = df.drop("career_aspiration", axis=1)
y = df["career_aspiration"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Encode labels
print("\n=== Label Encoding ===")
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Train model
print("\n=== Model Training ===")
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)
model.fit(x_train, y_train_enc)

# Evaluate
print("\n=== Evaluation ===")
print("Test Accuracy:", model.score(x_test, y_test_enc))

# Baseline comparison
print("\n=== Baseline Comparison ===")
dummy = DummyClassifier(strategy='stratified')
dummy.fit(x_train, y_train_enc)
print("Dummy Classifier Accuracy:", dummy.score(x_test, y_test_enc))

# Feature importance
print("\n=== Feature Importance ===")
fi = pd.DataFrame({
    'Feature': x_train.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)
print(fi)

# Confusion matrix (top 10 classes)
print("\n=== Confusion Matrix Preview ===")
y_pred = model.predict(x_test)
cm = confusion_matrix(y_test_enc, y_pred)
classes = le.classes_[:10]  # Show first 10 classes for brevity
class_idx = [list(le.classes_).index(c) for c in classes]
print(pd.DataFrame(
    cm[np.ix_(class_idx, class_idx)],
    columns=classes,
    index=classes
))

# Classification report
print("\n=== Classification Report ===")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))

# Data leakage check
print("\n=== Data Leakage Check ===")
print("Duplicate samples between train/test:", len(set(x_train.index) & set(x_test.index)))

# Save outputs to text file
with open('model_diagnostics.txt', 'w') as f:
    f.write("=== Feature Importance ===\n")
    f.write(fi.to_string())
    f.write("\n\n=== Classification Report ===\n")
    f.write(classification_report(y_test_enc, y_pred, target_names=le.classes_))

print("\n=== Diagnostic Complete ===")
print("Outputs saved to 'model_diagnostics.txt'")

=== Data Loading ===

=== Feature Engineering ===

=== Data Splitting ===

=== Label Encoding ===

=== Model Training ===

=== Evaluation ===
Test Accuracy: 0.96

=== Baseline Comparison ===
Dummy Classifier Accuracy: 0.09666666666666666

=== Feature Importance ===
                       Feature  Importance
6                biology_score    0.121976
2                   math_score    0.121026
9                   stem_score    0.109946
7                english_score    0.103917
4                physics_score    0.103818
3                history_score    0.103661
10            humanities_score    0.101403
5              chemistry_score    0.097897
8              geography_score    0.095083
0                       gender    0.024609
1   extracurricular_activities    0.016663

=== Confusion Matrix Preview ===
                       Accountant  Artist  Banker  Business Owner  \
Accountant                     78       0       0               0   
Artist                          0      36     

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

# 1. Load Dataset 1 from Excel
print("Loading Dataset 1...")
df = pd.read_excel("1.xlsx", sheet_name="original")

# 2. Data Preparation - Keep only intelligence scores and profession
print("\nPreparing data...")
# Corrected column names based on the df variable
intelligence_cols = ['Linguistic', 'Musical', 'Bodily', 'Logical - Mathematical',
                    'Spatial-Visualization', 'Interpersonal', 'Intrapersonal', 'Naturalist']
df = df[intelligence_cols + ['Job profession']] # Corrected column name

# 3. Feature Engineering - Create composite scores
print("\nEngineering features...")
df['Analytical_Score'] = (df['Logical - Mathematical'] + df['Spatial-Visualization']) / 2
df['Creative_Score'] = (df['Musical'] + df['Bodily']) / 2
df['Social_Score'] = (df['Interpersonal'] + df['Intrapersonal']) / 2

# 4. Split Data
print("\nSplitting data...")
X = df.drop('Job profession', axis=1) # Corrected column name
y = df['Job profession'] # Corrected column name

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Encode Labels
print("\nEncoding labels...")
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# 6. Train Model
print("\nTraining Random Forest...")
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=None,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train_enc)

# 7. Evaluate
print("\n=== Evaluation ===")
print("Test Accuracy:", model.score(X_test, y_test_enc))
print("\nClassification Report:")
print(classification_report(y_test_enc, model.predict(X_test),
      target_names=le.classes_))

# 8. Feature Importance
print("\nFeature Importances:")
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance)

# 9. Save Model
print("\nSaving model...")
joblib.dump(model, 'career_model_intelligence_only.pkl')
joblib.dump(le, 'label_encoder_intelligence.pkl')

print("\n=== Model Training Complete ===")
print("Saved files:")
print("- career_model_intelligence_only.pkl")
print("- label_encoder_intelligence.pkl")

Loading Dataset 1...
DataFrame columns: Index(['Sr.No.', 'Course', 'Job profession', 'Student', 'Linguistic',
       'Musical', 'Bodily', 'Logical - Mathematical', 'Spatial-Visualization',
       'Interpersonal', 'Intrapersonal', 'Naturalist', 's/p', 'P1', 'P2', 'P3',
       'P4', 'P5', 'P6', 'P7', 'P8'],
      dtype='object')

Preparing data...

Engineering features...

Splitting data...

Encoding labels...

Training Random Forest...

=== Evaluation ===
Test Accuracy: 0.9805555555555555

Classification Report:
                                                                                                precision    recall  f1-score   support

                                                                               Actor / Actress       1.00      1.00      1.00        10
                                                                                       Actuary       1.00      1.00      1.00        10
                                                                          

In [1]:
import joblib
import pandas as pd

def load_models():
    """Load the trained model and label encoder"""
    model = joblib.load('career_model_intelligence_only.pkl')
    le = joblib.load('label_encoder_intelligence.pkl')
    return model, le

def get_user_input():
    """Collect user's intelligence scores"""
    print("Please enter your scores (1-10) for each intelligence:")
    scores = {
        'Linguistic': float(input("Linguistic (verbal skills): ")),
        'Musical': float(input("Musical ability: ")),
        'Bodily': float(input("Bodily-kinesthetic (physical coordination): ")),
        'Logical - Mathematical': float(input("Logical-Mathematical: ")),
        'Spatial-Visualization': float(input("Spatial-Visualization: ")),
        'Interpersonal': float(input("Interpersonal (social skills): ")),
        'Intrapersonal': float(input("Intrapersonal (self-awareness): ")),
        'Naturalist': float(input("Naturalist (nature understanding): "))
    }

    # Calculate composite scores
    scores['Analytical_Score'] = (scores['Logical - Mathematical'] + scores['Spatial-Visualization']) / 2
    scores['Creative_Score'] = (scores['Musical'] + scores['Bodily']) / 2
    scores['Social_Score'] = (scores['Interpersonal'] + scores['Intrapersonal']) / 2

    # Convert to DataFrame with correct column order
    feature_order = [
        'Linguistic', 'Musical', 'Bodily', 'Logical - Mathematical',
        'Spatial-Visualization', 'Interpersonal', 'Intrapersonal', 'Naturalist',
        'Analytical_Score', 'Creative_Score', 'Social_Score'
    ]
    return pd.DataFrame([scores], columns=feature_order)

def predict_career(model, le, input_data):
    """Make and format predictions"""
    probabilities = model.predict_proba(input_data)[0]
    top_n = 5

    # Get top predictions
    top_indices = probabilities.argsort()[-top_n:][::-1]
    results = []
    for i in top_indices:
        results.append({
            'Career': le.classes_[i],
            'Probability': f"{probabilities[i]*100:.1f}%"
        })

    return results

def main():
    try:
        # Load models
        model, le = load_models()

        # Get user input
        print("\n=== Career Recommendation System ===")
        user_data = get_user_input()

        # Make prediction
        predictions = predict_career(model, le, user_data)

        # Display results
        print("\n=== Top Career Recommendations ===")
        for i, pred in enumerate(predictions, 1):
            print(f"{i}. {pred['Career']} ({pred['Probability']})")

    except Exception as e:
        print(f"\nError: {str(e)}")
        print("Please check:")
        print("- Model files exist in same directory")
        print("- All scores entered as numbers between 1-10")

if __name__ == "__main__":
    main()


Error: [Errno 2] No such file or directory: 'career_model_intelligence_only.pkl'
Please check:
- Model files exist in same directory
- All scores entered as numbers between 1-10


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

def main():
    try:
        # 1. Load Dataset
        print("Loading Dataset...")
        df = pd.read_excel("1.xlsx", sheet_name="original")

        # Verify required columns exist
        required_cols = ['Linguistic', 'Musical', 'Bodily', 'Logical - Mathematical',
                        'Spatial-Visualization', 'Interpersonal', 'Intrapersonal',
                        'Naturalist', 'Job profession']

        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")

        # 2. Data Preparation
        print("\nPreparing data...")
        df = df[required_cols].copy()

        # 3. Feature Engineering
        print("\nEngineering features...")
        df['Analytical_Score'] = (df['Logical - Mathematical'] + df['Spatial-Visualization']) / 2
        df['Creative_Score'] = (df['Musical'] + df['Bodily']) / 2
        df['Social_Score'] = (df['Interpersonal'] + df['Intrapersonal']) / 2

        # 4. Split Data
        print("\nSplitting data...")
        X = df.drop('Job profession', axis=1)
        y = df['Job profession']

        # Verify we have enough samples per class
        min_samples = y.value_counts().min()
        if min_samples < 5:
            raise ValueError(f"Some careers have too few samples (min={min_samples})")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y)

        # 5. Encode Labels
        print("\nEncoding labels...")
        le = LabelEncoder()
        y_train_enc = le.fit_transform(y_train)
        y_test_enc = le.transform(y_test)

        # 6. Train Model
        print("\nTraining Random Forest...")
        model = RandomForestClassifier(
            n_estimators=150,
            max_depth=None,
            min_samples_split=5,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1  # Use all CPU cores
        )
        model.fit(X_train, y_train_enc)

        # 7. Evaluate
        print("\n=== Evaluation ===")
        print(f"Test Accuracy: {model.score(X_test, y_test_enc):.4f}")

        print("\nClassification Report:")
        print(classification_report(y_test_enc, model.predict(X_test),
              target_names=le.classes_))

        # 8. Feature Importance
        print("\nFeature Importances:")
        feature_importance = pd.DataFrame({
            'Feature': X.columns,
            'Importance': model.feature_importances_
        }).sort_values('Importance', ascending=False)
        print(feature_importance.to_string())

        # 9. Save Model
        print("\nSaving model...")
        joblib.dump(model, 'career_model.pkl')
        joblib.dump(le, 'label_encoder.pkl')

        print("\n=== Model Training Complete ===")
        print("Saved files:")
        print("- career_model.pkl")
        print("- label_encoder.pkl")

    except Exception as e:
        print(f"\nError: {str(e)}")
        print("Please check:")
        print("- Excel file exists and has the correct sheet name")
        print("- All required columns are present")
        print("- There are enough samples for each career")

if __name__ == "__main__":
    main()

Loading Dataset...

Preparing data...

Engineering features...

Splitting data...

Encoding labels...

Training Random Forest...

=== Evaluation ===
Test Accuracy: 0.9806

Classification Report:
                                                                                                precision    recall  f1-score   support

                                                                               Actor / Actress       1.00      1.00      1.00        10
                                                                                       Actuary       1.00      1.00      1.00        10
                                                                                Anthropologist       1.00      1.00      1.00        10
                                                                                  Archeologist       1.00      1.00      1.00        10
                                                                                        Artist       1.00      1.00      1.0